In [83]:
import numpy as np
import cvxpy as cp
import mosek
import random
import matplotlib.pyplot as plt
from itertools import chain, combinations

In [84]:
def powerset(iterable):
    "powerset([1,2,3]) --> () (1,) (2,) (3,) (1,2) (1,3) (2,3) (1,2,3)"
    s = list(iterable)
    return chain.from_iterable(combinations(s, r) for r in range(len(s)+1))

def h_quad(x,m):
    return((1+m)*x-m*x**2)

def ranktoset (A):
    A = list(A)
    sets = [[A[0]]]
    for i in range(1,len(A)):
        new = A[0:i+1]
        sets.append(new)
    return(sets)

def makesetflex (A, B):    # we assume A is non-empty
    N = len(A)
    B = list(B)
    M = len(B)
    added = []
    for i in range(M):
        new = B[0:i+1]
        for k in range(N):
            if len(A[k])==len(new) and len(np.intersect1d(A[k],new))==len(new):
                break
            if k == N-1:
                A.append(new)
                added.append(new)
    return(A,added)

def robust_counterpart (sets,p,R,r,m,r_f,c):
    N = len(p)
    I = len(R[0])
    M = len(sets)
    v = cp.Variable(M)
    lbda = cp.Variable(M, nonneg = True)
    a = cp.Variable(I)
    alpha = cp.Variable(1)
    beta = cp.Variable(1)
    gamma = cp.Variable(1,nonneg = True)
    t = cp.Variable(N)
    w = cp.Variable(N, nonneg = True)
    z = cp.Variable(M)
    s = cp.Variable(N)
    eta = cp.Variable(M, nonneg= True)
    constraints = [s/2+gamma*(np.zeros(N)+1)<= w]
    for i in range(N):
        lbdasum = 0
        vsum = 0
        for j in range(M):
            if i in sets[j]:
                lbdasum = lbdasum + lbda[j]
                vsum = vsum + v[j]
        constraints.append((-R @ a)[i] - lbdasum - beta - (1-cp.sum(a))*r_f <= 0)
        constraints.append(s[i] == -alpha + vsum)
        constraints.append(cp.norm(cp.vstack([w[i],t[i]/2]))<=(t[i]+2*gamma)/2)
    for j in range(M):
        constraints.append(cp.norm(cp.vstack([eta[j],(z[j]-lbda[j])/2]))<=(z[j]+lbda[j])/2)
        constraints.append(1/(2*np.sqrt(m))*(-v[j]+lbda[j]+m*lbda[j])<= eta[j])
    constraints.append(cp.abs(a)<= 10)
    constraints.append(alpha + beta + gamma * r  + cp.sum(z) + p@t <= c)
    obj = cp.Maximize((R@a).T @ p + (1-cp.sum(a))*r_f)
    prob = cp.Problem(obj,constraints)
    prob.solve(solver=cp.MOSEK)
    return(a.value, prob.value)
    
def robustcheck(a,R,r,p,m,r_f):
    N = len(p)
    x = -R.dot(a)
    rank = np.argsort(R.dot(a))
    q_b = cp.Variable(N, nonneg = True)
    q = cp.Variable(N, nonneg=True)
    constraints = [cp.sum(q) == 1, cp.sum(q_b)==1]
    phi_cons = 0
    for i in range(N):
        z1 = q_b[rank[0:i+1]]
        z2 = q[rank[0:i+1]]
        v = (1+m)*cp.sum(z2)-m*cp.sum(z2)**2
        constraints.append(cp.sum(z1)-v <= 0)
        phi_cons = phi_cons +1/p[i]*(q[i]-p[i])**2
    constraints.append(phi_cons <= r)
    obj = cp.Maximize(q_b.T @ x)
    prob = cp.Problem(obj,constraints)
    prob.solve(solver=cp.MOSEK)
    return(prob.value - (1-np.sum(a))*r_f,q_b.value)

In [97]:
def squeeze_algo(R,r,c,p,m,r_f):
    N = len(p)
    I = len(R[0])
    a = cp.Variable(I)
    constraints = [cp.abs(a)<=10]
    h = np.zeros(N)
    iterations = 0
    steps = 0
    for i in range(N-1):
        h[i] = h_quad(sum(p[i:N]),m)-h_quad(sum(p[i+1:N]),m)
    h[N-1]=h_quad(p[N-1],m)
    constraints.append(-h.T@(R @ a)-(1-cp.sum(a))*r_f<= c)
    obj = cp.Maximize((R@a).T @ p + (1-cp.sum(a))*r_f)
    prob = cp.Problem(obj,constraints)
    prob.solve(solver=cp.MOSEK)
    w = a.value
    upperobj = prob.value
    oldrank = np.argsort(R.dot(w))
    sets = ranktoset(oldrank)
    nonstop = True
    nonstop2 = False
    firsttime = True
    lowerobj = -np.inf
    while nonstop:
        [rbvalue,h] = robustcheck(w,R,r,p,m,r_f)
        print('rbvalue',rbvalue)
        if rbvalue <= c:
            return('cut-stop',w,'upperbound', upperobj, 'lowerbound', lowerobj, 'cut-iterations', iterations,' robust iterations' ,steps)
        constraints.append(-h.T@(R @ a)-(1-cp.sum(a))*r_f<= c)
        iterations = iterations + 1
        if firsttime and rbvalue - c < 5:
            oldrank = np.argsort(R.dot(w))
            sets = ranktoset(oldrank)
            nonstop2 = True
            firsttime = False
        while nonstop2:
            [w,lowerobj] = robust_counterpart(sets,p,R,r,m,r_f,c)
            newrank = np.argsort(R.dot(w))
            if np.array_equal(newrank,oldrank):
                break
            [sets,added] = makesetflex(sets,newrank)
            oldrank = newrank
            steps = steps + 1
            print('RC steps',steps)
        if upperobj - lowerobj <= 1e-5:
            return('gap stop', w,'upperbound' , upperobj, 'lowerbound', lowerobj, 'cut-iterations', iterations,' robust iterations' ,steps)
        prob = cp.Problem(obj,constraints)
        prob.solve(solver=cp.MOSEK)
        w = a.value
        upperobj = prob.value
        if firsttime == False:
            newrank = np.argsort(R.dot(w))
            if np.array_equal(newrank,oldrank):
                pass
            else:
                lowerobj_new = robust_counterpart(ranktoset(newrank),p,R,r,m,r_f,c)[1]
                if lowerobj_new > lowerobj + 1e-5:
                    [sets,added] = makesetflex(sets,newrank)
                    oldrank = newrank
                    steps = steps + 1
                    print('RC steps',steps)
        print('upperbound' , upperobj, 'lowerbound', lowerobj, 'cut-iterations', iterations,' robust iterations' ,steps)

In [86]:
def normal_cutting_plane(R,r,c,p,m,r_f):
    N = len(p)
    I = len(R[0])
    a = cp.Variable(I)
    constraints = [cp.abs(a)<=10]
    h = np.zeros(N)
    iterations = 0
    for i in range(N-1):
        h[i] = h_quad(sum(p[i:N]),m)-h_quad(sum(p[i+1:N]),m)
    h[N-1]=h_quad(p[N-1],m)
    constraints.append(-h.T@(R @ a)-(1-cp.sum(a))*r_f<= c)
    obj = cp.Maximize((R@a).T @ p + (1-cp.sum(a))*r_f)
    prob = cp.Problem(obj,constraints)
    prob.solve(solver=cp.MOSEK)
    w = a.value
    upperobj = prob.value
    oldrank = np.argsort(R.dot(w))
    sets = ranktoset(oldrank)
    nonstop = True
    while nonstop:
        [rbvalue,h] = robustcheck(w,R,r,p,m,r_f)
        print('rbvalue',rbvalue,'iterations',iterations)
        if rbvalue <= c:
            return('cut-stop',w,'upperbound','obj',upperobj,'iterations',iterations)
        constraints.append(-h.T@(R @ a)-(1-cp.sum(a))*r_f<= c)
        iterations = iterations + 1
        prob = cp.Problem(obj,constraints)
        prob.solve(solver=cp.MOSEK)
        w = a.value
        upperobj = prob.value 

In [87]:
np.random.seed(5)

In [88]:
N=30
p = np.zeros(N)+1/N
I = 3
R = np.random.normal(0.05,0.2,size=(N,I))
print(R.transpose().dot(p))
#print(R)

[0.07174797 0.07208475 0.06633768]


In [89]:
r = 0.3
m = 0.9    # this is the parameter of the h function min(1, p/(1-m))
r_f = 0.001
c = 0.12

In [52]:
normal_cutting_plane(R,r,c,p,m,r_f)

rbvalue 0.9154716492909927 iterations 0
rbvalue 0.8694064958206542 iterations 1
rbvalue 1.2118357052972744 iterations 2
rbvalue 0.25636293160235846 iterations 3
rbvalue 0.17134673507415799 iterations 4
rbvalue 0.43891759945218156 iterations 5
rbvalue 0.17909815502176663 iterations 6
rbvalue 0.13577357157018982 iterations 7
rbvalue 0.13078721727566645 iterations 8
rbvalue 0.12396021517402847 iterations 9
rbvalue 0.12241226572383093 iterations 10
rbvalue 0.12110029058606919 iterations 11
rbvalue 0.12057991442723301 iterations 12
rbvalue 0.12018484312123058 iterations 13
rbvalue 0.12011302214484187 iterations 14
rbvalue 0.12006896554915487 iterations 15
rbvalue 0.12003280347191639 iterations 16
rbvalue 0.12001197427921997 iterations 17
rbvalue 0.12001010931292948 iterations 18
rbvalue 0.12000756663673383 iterations 19
rbvalue 0.12000252393851198 iterations 20
rbvalue 0.12000031869475564 iterations 21
rbvalue 0.11999989690442878 iterations 22


('cut-stop',
 array([1.69173466, 1.72679896, 0.98134903]),
 'upperbound',
 'obj',
 0.3075549357758954,
 'iterations',
 22)

In [98]:
squeeze_algo(R,r,c,p,m,r_f)

rbvalue 0.9154716492909927
RC steps 1
RC steps 2
RC steps 3
RC steps 4
upperbound 1.388526351801738 lowerbound 0.29142573026088925 cut-iterations 1  robust iterations 4
rbvalue 0.8694064958206542
upperbound 0.6060972398720493 lowerbound 0.29142573026088925 cut-iterations 2  robust iterations 4
rbvalue 1.2118357052972744
upperbound 0.45812035420932 lowerbound 0.29142573026088925 cut-iterations 3  robust iterations 4
rbvalue 0.25636293160235846
RC steps 5
upperbound 0.4258784954696432 lowerbound 0.29142573026088925 cut-iterations 4  robust iterations 5
rbvalue 0.17134673507415799
RC steps 6
RC steps 7
upperbound 0.35324820263849616 lowerbound 0.3075558581261816 cut-iterations 5  robust iterations 7
rbvalue 0.43891759945218156
upperbound 0.3362141447420344 lowerbound 0.3075558581261816 cut-iterations 6  robust iterations 7
rbvalue 0.17909815502176663
upperbound 0.3302547274575135 lowerbound 0.3075558581261816 cut-iterations 7  robust iterations 7
rbvalue 0.13577357157018982
upperbound 0.3

('gap stop',
 array([1.69258842, 1.72491284, 0.98249071]),
 'upperbound',
 0.3075552618376573,
 'lowerbound',
 0.3075558581261816,
 'cut-iterations',
 21,
 ' robust iterations',
 7)

In [44]:
y1= cp.Variable(1)
y2 = cp.Variable(1)
y3 = cp.Variable(1)
mylist = [y1, y2-y3]
z = cp.vstack(mylist)
x = cp.norm(z)
x.curvature

'CONVEX'

In [31]:
np.concatenate((1,2),axis=None)

array([1, 2])